/

## 20221366 박지선 자율이동체 중간 과제 보고서

## 1. 서론

 본 보고서는 KITTI odometry sequence 09 데이터를 이용하여 projection matrix, pose, Bayesian 기반 도로 영역 분류 결과를 분석한다. 특히 3차원 공간의 점이 이미지 평면으로 투영되는 과정과 차량의 이동 궤적을 시각화하고, Bayesian 분류로 얻은 도로 영역을 차선 후보 및 소실점 개념과 연관 지어 해석하는 것을 목적으로 한다.

 또한 Bayesian 방식이 조명 변화, 도로 질감 변화, 차량 회전 및 ROI 가정의 한계로 인해 실패할 수 있는 구간을 전체 sequence 09 프레임에서 찾고, 이를 pose 기반 궤적 위에 표시한다. 마지막으로 딥러닝 기반 semantic segmentation 모델인 DeepLabV3를 실패 후보 프레임에 적용하여 Bayesian 결과와 비교하고, 전용 도로/차선 segmentation 모델 학습의 필요성을 논의한다.

## 2. 실험 데이터 및 환경

- 데이터셋: KITTI odometry sequence 09
- 사용 이미지: image_0, grayscale camera, 전체 1591프레임
- 사용 보정 파일: calib.txt의 P0 projection matrix
- 사용 pose 파일: poses/09.txt
- Bayesian 출력 경로: outputs/bayes_road_sequence09_full
- 딥러닝 비교 모델: torchvision pretrained DeepLabV3-ResNet50
- 사용 언어 및 라이브러리: Python, NumPy, Matplotlib, PIL, PyTorch, torchvision
- 도로 분류 방식: Bayesian 기반 픽셀 밝기 히스토그램 분류


## 3. 본론

### 문제 1. Projection Matrix 해석

KITTI 데이터셋의 calib.txt 파일에서 sequence 09의 P0 projection matrix를 읽어 사용한다. P0는 rectified left grayscale camera 기준에서 3D 점을 2D 이미지 평면으로 투영하는 3x4 행렬이다.

Projection matrix P는 일반적으로 다음과 같이 표현된다.

P = K [R | t]

여기서 K는 intrinsic matrix이고, [R | t]는 카메라 좌표계와 기준 좌표계 사이의 extrinsic 관계를 나타낸다. 다만 KITTI odometry의 P0는 rectified camera 기준 projection matrix이므로, 본 실험에서는 P0의 좌측 3x3 부분을 통해 focal length와 principal point를 확인하고, 3D 점을 이미지 좌표로 투영하는 데 사용한다.

- **Intrinsic 파라미터 (f_x, f_y, c_x, c_y)**:
  - f_x, f_y: 픽셀 단위의 초점 거리.
  - c_x, c_y: principal point, 즉 이미지 중심에 가까운 기준점.

- **3D -> 2D 변환 수식**:
  3D 점 X = [X, Y, Z, 1]^T에 대해,

  [u, v, w]^T = P X

  u' = u / w, v' = v / w

아래 코드로 P0를 로드하고 intrinsic 파라미터를 확인한다.


In [ ]:
# calib.txt 읽기
from pathlib import Path
import numpy as np

calib_path = Path("dataset/sequences/09/calib.txt")
with open(calib_path, 'r') as f:
    lines = f.readlines()

# P0 추출 (첫 번째 행)
P0_line = lines[0].strip().split(' ')[1:]  # 'P0:' 이후
P0 = np.array([float(x) for x in P0_line]).reshape(3, 4)

print("Projection Matrix P0:")
print(P0)

# Intrinsic 파라미터 추출
K = P0[:, :3]
f_x = K[0, 0]
f_y = K[1, 1]
c_x = K[0, 2]
c_y = K[1, 2]

print(f"\nIntrinsic parameters:")
print(f"f_x (focal length x): {f_x}")
print(f"f_y (focal length y): {f_y}")
print(f"c_x (principal point x): {c_x}")
print(f"c_y (principal point y): {c_y}")

print("\nNote:")
print("P0 is used as the rectified camera projection matrix.")
print("Camera extrinsics are not separately estimated in this experiment.")


### 문제 2. Projection Matrix를 이용한 3D → 2D 투영

임의의 3D 점들을 생성하고, projection matrix P0를 이용하여 이미지 좌표로 투영한다.

투영된 점들을 이미지 위에 시각화한 자료는 아래 이미지와 같으며, 카메라 투영의 특성은 다음과 같다.

카메라 투영의 특성:
- 가까운 점은 크게, 먼 점은 작게 보임 (perspective projection).
- 소실점 방향으로 점들이 수렴.
- Z 좌표가 클수록 이미지에서 작아짐.

![3D to 2D Projection](outputs/sequence09_projection_demo.png)

**그림 1. Projection Matrix를 이용한 3D 점의 2D 이미지 평면 투영 결과**

In [ ]:
# 임의의 3D 점 생성 (도로 평면 상의 점들)
points_3d = np.array([
    [0, 0, 10, 1],  # 가까운 점
    [5, 0, 10, 1],
    [0, 5, 10, 1],
    [0, 0, 20, 1],  # 먼 점
    [10, 0, 20, 1],
    [0, 10, 20, 1]
])

# 투영
projected = P0 @ points_3d.T
u = projected[0] / projected[2]
v = projected[1] / projected[2]

print("3D Points:")
print(points_3d[:, :3])
print("\nProjected 2D Points (u, v):")
for i in range(len(u)):
    print(f"Point {i}: ({u[i]:.2f}, {v[i]:.2f})")

# 시각화 (텍스트 기반)
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.scatter(u, v, c='red', s=50)
for i, (x, y) in enumerate(zip(u, v)):
    plt.text(x, y, f'P{i}', fontsize=12)
plt.xlabel('u (pixels)')
plt.ylabel('v (pixels)')
plt.title('Projected 3D Points on Image Plane')
plt.gca().invert_yaxis()  # 이미지 좌표계
plt.show()

print("\n설명: 가까운 점(Z=10)은 크게, 먼 점(Z=20)은 작게 투영됨. 카메라 투영의 원근법 특성을 보여줌.")

### 문제 3. Pose를 이용한 차량 궤적 시각화

KITTI pose 데이터를 이용하여 차량의 이동 궤적을 시각화한다.

- 각 프레임의 카메라 위치를 추출 (t 벡터).
- 초기 프레임을 기준으로 상대 위치 계산.
- 2D 또는 3D로 표현.

Pose 데이터에서 각 프레임의 translation 성분을 추출하여 차량의 위치 변화를 확인하였다. 본 실험에서는 X-Z 평면을 기준으로 차량 궤적을 시각화하였으며, X축은 좌우 방향 이동, Z축은 전방 진행 방향의 이동을 의미한다.

궤적을 보면 차량은 일정한 직선 경로만 따라 이동하지 않고, 여러 구간에서 곡선 주행과 회전이 나타난다. 특히 Z 방향으로 위치 변화가 크게 나타나는 구간은 차량이 전방으로 주행한 구간으로 해석할 수 있으며, X 방향 변화가 함께 커지는 구간은 좌회전 또는 우회전과 같은 방향 변화가 발생한 구간으로 볼 수 있다.

차량의 속도는 연속된 두 프레임 사이의 위치 변화량을 이용하여 상대적으로 추정할 수 있다. 즉, i번째 프레임과 i+1번째 프레임의 위치 차이를 계산하면 프레임 간 이동 거리를 구할 수 있으며, 이 값이 클수록 해당 구간에서 차량이 더 빠르게 이동한 것으로 해석할 수 있다. 반대로 이동 거리가 작게 나타나는 구간은 감속, 회전, 또는 정지에 가까운 움직임이 포함되었을 가능성이 있다.

따라서 sequence 09의 차량 궤적은 전방 이동뿐만 아니라 좌우 방향 변화와 회전이 함께 포함된 주행 경로이며, 프레임 간 위치 변화량을 통해 차량의 상대적인 속도 변화도 확인할 수 있다.


![Vehicle Trajectory](outputs/sequence09_trajectory.png)

**그림 2. KITTI sequence 09 pose 데이터를 이용한 차량 이동 궤적**

In [ ]:
# poses 읽기
poses_path = Path("dataset 3/poses/09.txt")
poses = []
with open(poses_path, 'r') as f:
    for line in f:
        pose = np.array([float(x) for x in line.strip().split()]).reshape(3, 4)
        poses.append(pose)

# 각 프레임의 위치 추출 (t 벡터)
positions = [pose[:, 3] for pose in poses]

# 초기 프레임을 기준으로 상대 위치
initial_pos = positions[0]
relative_positions = [pos - initial_pos for pos in positions]

# 2D 시각화 (x, z 평면, y는 높이 무시)
x_coords = [pos[0] for pos in relative_positions]
z_coords = [pos[2] for pos in relative_positions]

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(x_coords, z_coords, marker='o', linestyle='-', markersize=2)
plt.xlabel('X (meters)')
plt.ylabel('Z (meters)')
plt.title('Vehicle Trajectory (Top View)')
plt.axis('equal')
plt.grid(True)
plt.show()

print("차량 궤적 설명: 차량은 주로 Z 방향(전진)으로 이동하며, X 방향으로 약간의 좌우 이동이 있음. 속도는 프레임 간 거리로 추정 가능.")

## 문제 4. Projection Matrix를 활용한 차선 해석

아래 코드는 Bayesian 분류 결과 마스크에서 도로 경계를 추출하고, 이미지 좌표에서 차선 후보를 해석하는 예이다.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

mask_path = Path("outputs/bayes_road_sequence09_full/mask_000000.png")
mask = Image.open(mask_path).convert("L")
mask_np = np.array(mask)
binary = (mask_np > 128).astype(np.uint8)

rows = np.arange(binary.shape[0] - 1, binary.shape[0] // 3, -5)
left_pts = []
right_pts = []
for y in rows:
    cols = np.where(binary[y] == 1)[0]
    if cols.size > 0:
        left_pts.append((cols[0], y))
        right_pts.append((cols[-1], y))

def fit_line(points):
    xs = np.array([p[0] for p in points], dtype=float)
    ys = np.array([p[1] for p in points], dtype=float)
    A = np.vstack([ys, np.ones_like(ys)]).T
    a, b = np.linalg.lstsq(A, rcond=None)[0]
    return a, b

left_a, left_b = fit_line(left_pts)
right_a, right_b = fit_line(right_pts)

plt.figure(figsize=(10, 6))
plt.imshow(mask_np, cmap="gray")
for pts, color, label in [(left_pts, "cyan", "left boundary"), (right_pts, "yellow", "right boundary")]:
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    plt.scatter(xs, ys, s=20, c=color, label=label)
ys_line = np.array([binary.shape[0] - 1, binary.shape[0] // 3])
plt.plot(left_a * ys_line + left_b, ys_line, "c-", linewidth=2)
plt.plot(right_a * ys_line + right_b, ys_line, "y-", linewidth=2)
plt.legend()
plt.title("Road mask boundaries and fitted lines")
plt.gca().invert_yaxis()
plt.show()

print("차선 해석:")
print("- 이미지 좌표에서 추출된 도로 경계는 카메라 좌표계에서 도로 평면 상의 경계에 대응합니다.")
print("- 도로를 평면으로 가정하면, 이미지 상의 직선은 3D 평면 위의 직선의 투영입니다.")
print("- Projection matrix는 3D 점을 이미지 픽셀로 매핑하며, 이때 경계선의 연장선은 소실점을 향할 수 있습니다.")

## 문제 5. 실패 구간 분석

Bayesian 도로 마스크의 픽셀 면적이 비정상적으로 작아지는 프레임을 실패 후보로 정의하였다. 이를 위해 sequence 09의 전체 1591프레임에 대해 생성한 마스크(`outputs/bayes_road_sequence09_full`)를 사용하고, 각 프레임의 도로 마스크 픽셀 수를 계산한 뒤 면적이 가장 작은 프레임을 우선적으로 검토한다.

마스크 면적이 작다는 것은 Bayesian 분류기가 도로 영역을 충분히 검출하지 못했다는 신호로 볼 수 있다. 이러한 실패 후보는 조명 변화, 그림자, 도로 재질 변화, 차량 회전 또는 곡선 도로로 인해 고정 ROI 가정이 맞지 않는 경우와 관련될 수 있다. 아래 코드는 전체 프레임 중 마스크 면적이 가장 작은 프레임을 찾고, 해당 위치를 pose 기반 궤적 위에 표시한다.


In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

mask_dir = Path("outputs/bayes_road_sequence09_full")
mask_files = sorted(mask_dir.glob("mask_*.png"))
areas = []
for p in mask_files:
    frame = int(p.stem.split("_")[1])
    mask = np.array(Image.open(p).convert("L"))
    areas.append((frame, (mask > 128).sum()))

areas = sorted(areas, key=lambda x: x[1])
print("도로 마스크 면적이 가장 작은 프레임:")
for frame, area in areas[:5]:
    print(f"Frame {frame}: pixel area {area}")

selected_frame = areas[0][0]
selected_mask = np.array(Image.open(mask_dir / f"mask_{selected_frame:06d}.png").convert("L"))

plt.figure(figsize=(8, 6))
plt.imshow(selected_mask, cmap="gray")
plt.title(f"Failed road mask at frame {selected_frame}")
plt.axis("off")
plt.show()

poses_path = Path("dataset 3/poses/09.txt")
poses = []
with open(poses_path, "r") as f:
    for line in f:
        pose = np.array([float(x) for x in line.strip().split()]).reshape(3, 4)
        poses.append(pose)

positions = [pose[:, 3] for pose in poses]
selected_pos = positions[selected_frame]
x_coords = [pos[0] - positions[0][0] for pos in positions]
z_coords = [pos[2] - positions[0][2] for pos in positions]

plt.figure(figsize=(10, 6))
plt.plot(x_coords, z_coords, marker="o", markersize=2)
plt.scatter([selected_pos[0] - positions[0][0]], [selected_pos[2] - positions[0][2]], c="red", s=50, label=f"Frame {selected_frame}")
plt.xlabel("X (meters)")
plt.ylabel("Z (meters)")
plt.title("Trajectory with failed frame highlighted")
plt.legend()
plt.axis("equal")
plt.grid(True)
plt.show()

print("선택된 실패 구간:", selected_frame)
print("- 도로 마스크 영역이 가장 작아진 프레임입니다.")
print("- 조명 변화/그림자: 밝기 대비가 낮아져 Bayesian 분류가 흔들릴 수 있습니다.")
print("- 도로 질감 변화: 차선이나 도로 색이 희미하면 도로/배경 구분이 어려워집니다.")
print("- 차량 회전/곡선: ROI 사다리꼴 가정이 실제 도로 형상을 충분히 잡지 못합니다.")

## 문제 6. 딥러닝 모델 제안 및 적용 결과 비교

차선/도로 영역 검출을 위한 딥러닝 모델로 semantic segmentation 계열의 **DeepLabV3**를 선택하였다. DeepLabV3는 convolutional neural network를 이용해 이미지의 각 픽셀을 클래스별로 분류하는 모델이며, 도로와 차선처럼 공간적 형태와 주변 문맥이 중요한 문제에 사용할 수 있다. Bayesian 방식이 픽셀 밝기 histogram만 사용하는 것과 달리, DeepLabV3는 edge, texture, object context, 주변 구조를 함께 반영할 수 있다는 장점이 있다.

본 실험에서는 torchvision에서 제공하는 pretrained DeepLabV3-ResNet50 모델을 sequence 09의 실패 후보 프레임(frame 632)에 적용하였다. 단, 해당 pretrained 모델은 VOC/COCO 계열 클래스에 학습되어 있어 `road` 또는 `lane` 클래스를 직접 포함하지 않는다. 따라서 이 결과는 완성된 차선 검출 결과라기보다, **일반 semantic segmentation 모델을 그대로 적용했을 때의 한계**와 **KITTI 도로/차선 데이터로 fine-tuning된 전용 모델이 필요한 이유**를 보여주는 비교 실험으로 해석한다.


![Bayesian and DeepLab comparison](outputs/deeplab_bayesian_comparison_000632.png)

**그림 3. Bayesian 도로 마스크와 pretrained DeepLabV3 적용 결과 비교(frame 632)**

DeepLabV3 적용 결과, 모델은 `background`, `car`, `person` 클래스를 예측하였다. 즉 pretrained 모델은 도로 또는 차선을 별도 클래스로 분류하지 못했으며, Bayesian 도로 마스크와 DeepLabV3 non-background 영역의 IoU는 약 0.018로 매우 낮게 나타났다. 이는 일반 객체 segmentation 모델을 그대로 사용하는 것만으로는 차선/도로 검출 문제를 해결하기 어렵다는 점을 보여준다.

| 방식 | 적용 결과 | 장점 | 한계 |
|---|---|---|---|
| Bayesian 밝기 기반 분류 | 도로 후보 영역을 넓게 검출 | 구현이 단순하고 빠르며 학습 데이터가 거의 필요 없음 | 조명, 그림자, 도로 재질 변화, ROI 가정에 취약 |
| Pretrained DeepLabV3 | car/person/background 위주로 검출 | 이미지의 공간적 문맥과 객체 형태를 반영 가능 | road/lane 클래스가 없어 차선 검출 결과로 직접 사용하기 어려움 |
| Fine-tuned DeepLabV3 또는 LaneNet | 도로/차선 라벨로 학습하면 직접적인 차선/도로 segmentation 가능 | 곡선, 그림자, 복잡한 도로 상황에 더 강할 수 있음 | 별도 라벨 데이터와 학습 과정 필요 |


### 결론

문제 6의 비교 결과를 통해 Bayesian 방식은 간단한 baseline으로 사용할 수 있지만, 실제 차선 검출에는 한계가 있음을 확인하였다. DeepLabV3와 같은 segmentation 모델은 구조적으로 차선/도로 검출에 적합하지만, pretrained VOC/COCO 모델에는 road/lane 클래스가 없으므로 KITTI road/lane 라벨 또는 유사한 주행 데이터셋으로 fine-tuning해야 한다. 따라서 최종적으로는 Bayesian 결과를 baseline으로 두고, fine-tuned DeepLabV3, U-Net, LaneNet 계열 모델을 적용하여 IoU와 정성적 결과를 비교하는 방향이 적절하다.

본 보고서에서는 frame 632에서 Bayesian 마스크와 pretrained DeepLabV3 결과를 비교하여, 전용 딥러닝 모델 학습의 필요성을 실험적으로 보였다.
